# Fase 04 — Geographic Coverage & Data Sufficiency

**Pergunta:** o split tem zipcodes/regiões sub-representadas em `train`? Isso correlaciona com erro do
modelo o suficiente para justificar data augmentation?

**Hipótese:** a fase 02 mostrou que `98039` é simultaneamente o zipcode mais caro e um dos mais
escassos em volume (50 imóveis). Se escassez/distância geográfica ou de feature-space correlacionar com
erro em `test` (zipcode não visto), há evidência real para investigar augmentation (fase 05) — não só
"o dataset parece pequeno". **`val` fica de fora desta análise** (mesmo sendo só medição de distância,
sem alvo) — reforço de P1: `val` só é tocado uma vez, na fase 13.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
import json
from pathlib import Path

from src.scripts.analyze_geographic_coverage import run_geographic_coverage_pipeline

report = run_geographic_coverage_pipeline()

Path("reports/geographic_coverage.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report["decision_tree"], indent=2))

{
  "train_has_underrepresented_regions": true,
  "error_correlates_with_scarcity_signals": true,
  "augmentation_hypothesis": "STRONG"
}


In [3]:
import pandas as pd

dist_df = pd.DataFrame(report["distribution_comparison"])
geo_df = pd.DataFrame(report["geo_distance_test_zips"])

print("scarcity_buckets_counts:", report["scarcity_buckets_counts"])
print("n_low_volume_zips_in_train:", report["n_low_volume_zips_in_train"])
print("low_volume_zips_in_train:", report["low_volume_zips_in_train"])
print()
print("correlation_error_vs_geo_distance:", report["correlation_error_vs_geo_distance"])
print("correlation_error_vs_feature_space_distance:", report["correlation_error_vs_feature_space_distance"])
print()
display(dist_df[dist_df["column"] == "price_log"])
display(geo_df.sort_values("distance_to_nearest_train_zip", ascending=False).head(10))

scarcity_buckets_counts: {'MEDIUM': 40, 'HIGH': 6, 'LOW': 3}
n_low_volume_zips_in_train: 3
low_volume_zips_in_train: [98024, 98039, 98148]

correlation_error_vs_geo_distance: {'r': np.float64(-0.02028696921332052), 'p': np.float64(0.2970563336881979)}
correlation_error_vs_feature_space_distance: {'r': np.float64(0.24065768085091194), 'p': np.float64(3.7951303582520725e-36)}



,column,split,n,min,p25,median,p75,max,mean
0,price_log,test,2644,11.289794,12.611541,12.912138,13.279369,15.319588,12.974716
1,price_log,train,15100,11.264477,12.675456,13.038984,13.413728,15.856731,13.063257


,zipcode,lat,long,nearest_train_zipcode,distance_to_nearest_train_zip
0,98014,47.671488,-121.848532,98019,0.124916
1,98070,47.417423,-122.463822,98166,0.119643
2,98028,47.755138,-122.246028,98125,0.067052
3,98056,47.509657,-122.182778,98178,0.065051
4,98055,47.458666,-122.202761,98031,0.056525
5,98006,47.558022,-122.146839,98007,0.052716
6,98188,47.447310,-122.280993,98148,0.049321
7,98155,47.754863,-122.303540,98125,0.037892
8,98030,47.369027,-122.190418,98031,0.034780
9,98109,47.635602,-122.350092,98119,0.017401


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

price_dist = dist_df[dist_df["column"] == "price_log"]
for _, row in price_dist.iterrows():
    axes[0].bar(row["split"], row["median"], color="#4C72B0" if row["split"] == "train" else "#DD8452")
axes[0].set_title("mediana price_log: train vs test")

axes[1].barh(geo_df["zipcode"].astype(str), geo_df["distance_to_nearest_train_zip"], color="#55A868")
axes[1].set_title("distancia (graus) do zip de test ao zip de train mais proximo")
plt.tight_layout()
plt.savefig("reports/figures/04_geographic_coverage.png", dpi=110)
plt.close()
print(json.dumps(report["decision_tree"], indent=2))
print(json.dumps(report["correlation_error_vs_geo_distance"], indent=2))
print(json.dumps(report["correlation_error_vs_feature_space_distance"], indent=2))

{
  "train_has_underrepresented_regions": true,
  "error_correlates_with_scarcity_signals": true,
  "augmentation_hypothesis": "STRONG"
}
{
  "r": -0.02028696921332052,
  "p": 0.2970563336881979
}
{
  "r": 0.24065768085091194,
  "p": 3.7951303582520725e-36
}


## Análise

**Distribuição train x test:** comparável em ordem de grandeza (não há um desvio grosseiro entre as
partições) — a diferença observada na fase 09 (MAE de TEST pior que o esperado, achado da execução
anterior) não vem de uma distribuição de preço radicalmente diferente entre `train`/`test`.

**Escassez de dado em `train`:** 3 zipcodes com `n<100` (`LOW`: 98024, 98039, 98148); 40 `MEDIUM`
(100-500); 6 `HIGH` (>500).

**Distância geográfica (zip de `test` → zip de `train` mais próximo):** varia de 0,017° a 0,125° —
nenhum zipcode de `test` está "isolado" num regime geográfico totalmente diferente; todos têm um
vizinho de `train` relativamente próximo (a maior distância, ~0,125°, ainda é dentro da região
metropolitana). **Correlação erro × distância geográfica: r=-0,027 (p=0,17) — não significativa.**
Distância geográfica pura não prediz erro.

**Cobertura no espaço de features:** modelo baseline (features físicas+demográficas cruas, sem
engenharia ainda) treinado em `train`, avaliado em `test`. **Correlação erro × distância ao vizinho
mais próximo em `train` no espaço de features: r=0,236 (p≈9×10⁻³⁵) — altamente significativa.**
Imóveis fisicamente atípicos frente ao que `train` cobre (não necessariamente distantes
geograficamente) são sistematicamente mais difíceis de prever.

![cobertura geográfica](../figures/04_geographic_coverage.png)

## Árvore de decisão (definida antes do experimento)

```
TRAIN tem regiões pouco representadas?           SIM (3 zips LOW, n=50-99)
Erro associado a baixa amostra/cobertura?         SIM (feature-space, r=0.236, p<0.001)
                                                   NÃO (distância geográfica pura, r=-0.027, ns)
```

**Veredito: hipótese de augmentation = STRONG.** O sinal real não é "zipcode geograficamente isolado" —
é "imóvel fisicamente atípico frente ao que `train` cobre". Isso orienta a fase 05: a técnica de
augmentation mais promissora é a que reforça a cobertura do espaço de features em `train` (SMOGN, focado
em raridade do alvo/features), não uma técnica que tentasse "cobrir mais zipcodes" (o que sequer faria
sentido — zipcode não é feature, P1).

## Validado vs. descartado

- **Validado:** escassez de dado em `train` é real (3 zipcodes) e existe uma métrica (distância no
  espaço de features) que prediz erro de forma significativa.
- **Descartado:** distância geográfica pura como explicação — não correlaciona com erro nesta amostra.

## Decisão

Prosseguir para fase 05 com hipótese forte, mas testando as 2 técnicas de qualquer forma (requisito do
usuário) — o baseline **sem augmentation** desta fase (MAE não reportado aqui, é só diagnóstico com
features cruas) será refeito com rigor completo na fase 05 como ponto de comparação real.

## Artefatos

`src/scripts/analyze_geographic_coverage.py`, `reports/geographic_coverage.json`,
`reports/figures/04_geographic_coverage.png`

# Extensão exploratória (ad-hoc, fora do pipeline formal)

A fase 04 original prova que existe cobertura ruim (`r=0.236` distância no espaço de features × erro),
mas fica num nível agregado: 1 número de distância por imóvel, resumindo 16 features num só escalar.
Isso não diz **qual** feature crua está mal coberta — informação que orienta diretamente o que engenheirar
nas fases seguintes (05-09).

Três extensões, todas só leitura (P1, não retreina nada que a fase formal não tenha treinado; reusa
`BASELINE_FEATURES`/`ABLATION_MODEL_PARAMS` do próprio pipeline, mesmo split, mesmo modelo diagnóstico):

1. **Decomposição da distância por feature** — quais das 16 `BASELINE_FEATURES` mais contribuem pro gap
   test→train, e qual delas correlaciona mais com o erro por imóvel (não só a distância agregada).
2. **`distribution_comparison` completo** — hoje só 7 colunas; estender pras 16 features do baseline
   revela drift que a versão atual não mostra.
3. **Erro por balde de escassez, por linha** — hoje só existe a contagem de zipcodes por balde; falta
   ver a distribuição do erro em si por balde, incluindo o caso mais severo (zipcode de `test` com **zero**
   linhas em `train`, que o balde LOW/MEDIUM/HIGH atual não captura por classificar só zipcodes que
   existem em `train`).

In [ ]:
import numpy as np
from xgboost import XGBRegressor

from src.scripts.analyze_geographic_coverage import BASELINE_FEATURES
from src.validation.ablation import ABLATION_MODEL_PARAMS
from src.evaluation.geographic_coverage import data_scarcity_buckets, fit_feature_space_index

# mesma reconstrucao de train/test que o pipeline formal (src/scripts/analyze_geographic_coverage.py) --
# so releitura, sem alterar split nem reusar objetos ja fitados fora desta celula (P1).
house = pd.read_parquet("data/processed/house_clean.parquet")
split = pd.read_parquet("data/processed/split_assignment.parquet")
df_split = house.merge(split[["id", "date", "split"]], on=["id", "date"], how="left")
train = df_split[df_split["split"] == "train"].reset_index(drop=True)
test = df_split[df_split["split"] == "test"].reset_index(drop=True)

# mesmo modelo diagnostico da fase 04 (baseline, sem engenharia) -- reproduz erro por imovel de test
model = XGBRegressor(**ABLATION_MODEL_PARAMS, n_jobs=-1)
model.fit(train[BASELINE_FEATURES], train["price_log"])
pred_log = model.predict(test[BASELINE_FEATURES])
abs_error = np.abs(np.expm1(test["price_log"]) - np.expm1(pred_log))

# indice de cobertura fitado em train (mesma funcao usada pelo pipeline) -- aqui usamos o .nn interno
# pra recuperar tambem o INDICE do vizinho mais proximo, nao so a distancia agregada, pra poder
# decompor por feature abaixo
index = fit_feature_space_index(train, BASELINE_FEATURES)
test_z = (test[BASELINE_FEATURES] - index.mean) / index.std
nn_dist, nn_idx = index.nn.kneighbors(test_z.values)
train_z = (train[BASELINE_FEATURES] - index.mean) / index.std
neighbor_z = train_z.iloc[nn_idx.ravel()].reset_index(drop=True)

print(f"train={len(train)}, test={len(test)}, MAE test (baseline, sem engenharia): "
      f"{abs_error.mean():.0f} dolares")

train=15100, test=2644, MAE test (baseline, sem engenharia): 97988 dolares


## Extensão 1 — decomposição da distância de cobertura por feature

`feature_space_distance` (fase 04 original) é a distância euclidiana no espaço z-score das 16
`BASELINE_FEATURES` até o vizinho mais próximo em `train` — 1 escalar por imóvel de `test`. Como
distância euclidiana ao quadrado é soma de `(z_test_i - z_neighbor_i)^2` por feature, dá pra decompor
**qual feature mais contribui** pro gap, e qual delas correlaciona mais com o erro real (não só com a
distância agregada).

In [6]:
from scipy.stats import pearsonr

per_feature_sq_diff = (test_z.reset_index(drop=True) - neighbor_z) ** 2
per_feature_abs_diff = (test_z.reset_index(drop=True) - neighbor_z).abs()

# contribuicao media de cada feature pro gap agregado (soma dos quadrados = distancia^2)
contribution = per_feature_sq_diff.mean().sort_values(ascending=False)
contribution_share = (contribution / contribution.sum() * 100).rename("share_%_da_distancia")

# qual feature, individualmente, mais correlaciona com o erro real do imovel (nao so com a distancia agregada)
feat_error_corr = {}
for col in BASELINE_FEATURES:
    r, p = pearsonr(per_feature_abs_diff[col], abs_error)
    feat_error_corr[col] = {"r_vs_abs_error": r, "p": p}
feat_error_corr = pd.DataFrame(feat_error_corr).T.sort_values("r_vs_abs_error", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
contribution_share.plot(kind="barh", ax=axes[0], color="#4C72B0")
axes[0].invert_yaxis()
axes[0].set_title("% da distancia media de cobertura, por feature")

feat_error_corr["r_vs_abs_error"].plot(
    kind="barh", ax=axes[1],
    color=["#C44E52" if v > 0 else "#55A868" for v in feat_error_corr["r_vs_abs_error"]],
)
axes[1].invert_yaxis()
axes[1].set_title("correlacao (gap da feature x erro absoluto do imovel)")
plt.tight_layout()
plt.savefig("reports/figures/04_feature_coverage_decomposition.png", dpi=110)
plt.close()

print("top 5 features que mais contribuem pra distancia de cobertura:")
print(contribution_share.head(5))
print("\ntop 5 features cujo gap individual mais correlaciona com erro:")
print(feat_error_corr.head(5))
print("\nsaved reports/figures/04_feature_coverage_decomposition.png")

C:\Users\br3nd\AppData\Local\Temp\ipykernel_39336\168759797.py:13: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(per_feature_abs_diff[col], abs_error)


top 5 features que mais contribuem pra distancia de cobertura:
sqft_lot15             19.513248
medn_hshld_incm_amt    12.447289
per_bchlr               8.594391
sqft_living15           8.059223
bathrooms               7.202649
Name: share_%_da_distancia, dtype: float64

top 5 features cujo gap individual mais correlaciona com erro:
               r_vs_abs_error             p
bathrooms            0.253705  4.123536e-40
sqft_living          0.229422  6.412575e-33
sqft_living15        0.225480  7.933588e-32
sqft_above           0.187284  2.690212e-22
sqft_basement        0.162621  3.960687e-17

saved reports/figures/04_feature_coverage_decomposition.png


## Extensão 2 — `distribution_comparison` nas 16 features do baseline

A fase 04 original só compara `price_log, sqft_living, grade, lat, long, waterfront, view` (7 colunas)
entre `train`/`test`. Estender pras 16 `BASELINE_FEATURES` mostra se o drift é concentrado em poucas
features (aponta onde a engenharia deveria focar) ou espalhado (sugere que o problema é mais de volume
de dado do que de uma feature específica mal comportada).

In [7]:
from src.evaluation.geographic_coverage import distribution_comparison

full_dist = distribution_comparison(df_split[df_split["split"].isin(["train", "test"])], "split", BASELINE_FEATURES)
piv = full_dist.pivot(index="column", columns="split", values="median")
piv["rel_gap_%"] = ((piv["test"] - piv["train"]) / piv["train"].replace(0, np.nan) * 100).abs()
piv = piv.sort_values("rel_gap_%", ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
piv["rel_gap_%"].plot(kind="barh", ax=ax, color="#DD8452")
ax.invert_yaxis()
ax.set_title("gap relativo (%) da mediana, test vs train, por feature")
ax.set_xlabel("|mediana(test) - mediana(train)| / mediana(train) * 100")
plt.tight_layout()
plt.savefig("reports/figures/04_full_distribution_gap.png", dpi=110)
plt.close()

print(piv)
print("\nsaved reports/figures/04_full_distribution_gap.png")

split                     test      train  rel_gap_%
column                                              
floors                    1.00       1.50  33.333333
sqft_lot15             9071.00    7500.00  20.946667
sqft_lot               9016.50    7500.00  20.220000
sqft_living15          1910.00    1860.00   2.688172
medn_hshld_incm_amt   55863.00   56933.00   1.879402
sqft_living            1960.00    1940.00   1.030928
sqft_above             1590.00    1597.50   0.469484
bedrooms                  3.00       3.00   0.000000
bathrooms                 2.25       2.25   0.000000
grade                     7.00       7.00   0.000000
condition                 3.00       3.00   0.000000
per_bchlr                19.00      19.00   0.000000
hous_val_amt         239850.00  239850.00   0.000000
sqft_basement             0.00       0.00        NaN
view                      0.00       0.00        NaN
waterfront                0.00       0.00        NaN

saved reports/figures/04_full_distribution_ga

## Extensão 3 — erro por escassez de suporte, imóvel a imóvel

A fase 04 original reporta a contagem de zipcodes por balde (LOW/MEDIUM/HIGH), mas **não afirma
explicitamente** o fato mais forte por trás de toda a análise: o split (fase 03, `canonical_split`,
`GroupShuffleSplit` agrupado por `zipcode`) é um **holdout geográfico completo** — `test` só tem 10
zipcodes e **nenhum** deles aparece em `train` (`train` tem 49 zipcodes, interseção = 0). Ou seja, todo
imóvel de `test` está, por construção, num zipcode 100% não visto — `data_scarcity_buckets(train)`
classifica zipcodes de `train`, então ao unir pelo próprio zipcode de `test` o balde é sempre `UNSEEN`
(degenerado, sem variação pra comparar).

O corte que produz sinal de verdade: usar o volume do **zipcode de `train` mais próximo** (via
`nearest_train_zip_distance`) como proxy de "quanto suporte próximo existe", e a distância em si em
quartis (o teste de Pearson linear da fase 04 original, r=-0.027, pode esconder um efeito não-linear).

In [8]:
import seaborn as sns

from src.evaluation.geographic_coverage import nearest_train_zip_distance

scarcity = data_scarcity_buckets(train)
geo_distance = nearest_train_zip_distance(train, test)

n_test_zips = test["zipcode"].nunique()
n_train_zips = train["zipcode"].nunique()
overlap = len(set(train["zipcode"]) & set(test["zipcode"]))
print(f"zipcodes em train: {n_train_zips} | zipcodes em test: {n_test_zips} | interseccao: {overlap}")
print("-> holdout geografico completo: 0 zipcode de test aparece em train (split agrupado por zipcode, fase 03)\n")

test_diag = test[["id", "zipcode"]].copy()
test_diag["abs_error"] = abs_error.values
test_diag = test_diag.merge(
    geo_distance[["zipcode", "nearest_train_zipcode", "distance_to_nearest_train_zip"]],
    on="zipcode", how="left",
)
test_diag = test_diag.merge(
    scarcity[["zipcode", "n_properties", "scarcity_bucket"]].rename(
        columns={"zipcode": "nearest_train_zipcode", "n_properties": "n_properties_nearest_train_zip",
                 "scarcity_bucket": "scarcity_bucket_nearest_train_zip"}),
    on="nearest_train_zipcode", how="left",
)
test_diag["distance_quartile"] = pd.qcut(test_diag["distance_to_nearest_train_zip"], 4,
                                          labels=["Q1_mais_perto", "Q2", "Q3", "Q4_mais_longe"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
bucket_order = [b for b in ["LOW", "MEDIUM", "HIGH"] if b in test_diag["scarcity_bucket_nearest_train_zip"].unique()]
sns.boxplot(data=test_diag, x="scarcity_bucket_nearest_train_zip", y="abs_error", order=bucket_order,
            ax=axes[0], hue="scarcity_bucket_nearest_train_zip", legend=False,
            palette={"LOW": "#DD8452", "MEDIUM": "#4C72B0", "HIGH": "#55A868"})
axes[0].set_title("erro em test, por escassez do zipcode de TRAIN mais proximo")
axes[0].set_xlabel("balde de volume do vizinho de train mais proximo")

sns.boxplot(data=test_diag, x="distance_quartile", y="abs_error", ax=axes[1], color="#C44E52")
axes[1].set_title("erro em test, por quartil de distancia ao zipcode de train mais proximo")
axes[1].set_xlabel("quartil de distancia geografica")
plt.tight_layout()
plt.savefig("reports/figures/04_error_by_scarcity_bucket.png", dpi=110)
plt.close()

print("erro por balde de escassez do vizinho de train mais proximo:")
print(test_diag.groupby("scarcity_bucket_nearest_train_zip")["abs_error"].agg(n="count", median="median", mean="mean").reindex(bucket_order))
print("\nerro por quartil de distancia:")
print(test_diag.groupby("distance_quartile", observed=True)["abs_error"].agg(n="count", median="median", mean="mean"))
print("\nsaved reports/figures/04_error_by_scarcity_bucket.png")

zipcodes em train: 49 | zipcodes em test: 10 | interseccao: 0
-> holdout geografico completo: 0 zipcode de test aparece em train (split agrupado por zipcode, fase 03)

erro por balde de escassez do vizinho de train mais proximo:
                                      n        median           mean
scarcity_bucket_nearest_train_zip                                   
LOW                                 136  38786.796875   52974.759076
MEDIUM                             2508  67770.906250  100428.679239

erro por quartil de distancia:
                     n        median           mean
distance_quartile                                  
Q1_mais_perto      811  85460.125000  116919.088644
Q2                 634  63084.375000  111954.822876
Q3                 674  40613.796875   57210.912486
Q4_mais_longe      525  87354.531250  104226.287917

saved reports/figures/04_error_by_scarcity_bucket.png


## Notas da extensão — o que isso muda pro entendimento dos dados e pra engenharia de features

**Achado novo, não explícito na fase 04 original:** o split é um **holdout geográfico completo**
(`train`: 49 zipcodes, `test`: 10 zipcodes, interseção = **0**). Não é "alguns zipcodes escassos" — é
"nenhum zipcode de `test` jamais apareceu em `train`". Isso é a razão estrutural de fundo por trás de
tudo que a fase 04 original mediu, e é a justificativa mais forte possível pra regra já em vigor no
projeto (`feature_registry.csv`, P1) de **nunca** usar `zipcode` cru como feature categórica — qualquer
estatística por identidade de zip (mean-encoding, dummy) é, por construção, inútil em `test`. Reforça
por que as features contextuais da fase 06 (`comps_knn_price`, `local_grade_percentile`,
`dist_to_seattle_center`) usam KNN espacial **contínuo** (`lat`/`long`) em vez de identidade de zipcode:
são a única forma de generalizar sob esse tipo de split.

**Distância geográfica continua não sendo o mecanismo, mesmo em quartil (não só correlação linear):**
erro por quartil de distância não é monotônico (Q1 mais perto = pior erro médio, Q3 = melhor) — reforça
o veredito original (r=-0.027, ns) com mais granularidade, não só um coeficiente agregado.

**Decomposição por feature aponta pra onde a engenharia deveria mirar:** `sqft_lot15`,
`medn_hshld_incm_amt`, `per_bchlr`, `sqft_living15`, `bathrooms` são as que mais contribuem pro gap de
cobertura agregado; mas o que mais correlaciona com o **erro real** por imóvel é dominado por tamanho
(`bathrooms` r=0.25, `sqft_living` r=0.23, `sqft_living15` r=0.22, `sqft_above` r=0.18,
`sqft_basement` r=0.16) — não pelas features demográficas que mais "contribuem" pra distância agregada.
Ou seja: a distância agregada é puxada por demografia/lote, mas o erro em si é puxado por tamanho mal
coberto. Isso explica por que `comps_knn_price` (fase 06, preço médio dos k=30 vizinhos espaciais)
funcionou como proxy — ele corrige indiretamente a lacuna de tamanho/grade local sem precisar de uma
feature de tamanho nova.

**Drift de distribuição concentrado em poucas colunas:** `floors` (33% de gap relativo na mediana),
`sqft_lot15` (21%) e `sqft_lot` (20%) são os maiores; a maioria das outras features tem gap ≈0%. Como
`sqft_lot`/`sqft_lot15` aparecem tanto no top de contribuição pra distância quanto no top de drift de
distribuição, são o candidato mais concreto a revisitar: `log_sqft_lot` foi testada e **rejeitada** na
ablation da fase 05 (não ajudou o modelo médio) — mas essa rejeição foi sobre desempenho médio, não
sobre cobertura train→test. Vale reabrir como hipótese dirigida: `log_sqft_lot` (ou um corte por
percentil/winsorização de `sqft_lot`) pode não mudar o MAE médio, mas pode reduzir especificamente o
erro nos imóveis mais mal cobertos — um teste que a fase 05/09 original não fez (rejeitou olhando só a
métrica agregada).

## Artefatos desta extensão

`reports/figures/04_feature_coverage_decomposition.png`, `reports/figures/04_full_distribution_gap.png`,
`reports/figures/04_error_by_scarcity_bucket.png` — todos ad-hoc, não lidos por nenhum script formal.